In [0]:
%pip install shapely

In [0]:
import json, math
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

CATALOG= "hawkerreach"

In [0]:
from pyspark.sql import functions as F

CATALOG= "hawkerreach"
raw= spark.table(f"{CATALOG}.bronze.raw_hawker_centres")
display(raw.groupBy("status").count())

In [0]:
KEEP_STATUS= ["Existing", "Existing (new)", "Existing (replacement)", "Interim Centre"]

centres= (raw
    .filter(F.col("status").isin(KEEP_STATUS))
    .withColumn("cooked_food_stalls", F.col("number_of_cooked_food_stalls").cast("int"))
    .filter(F.col("cooked_food_stalls") > 0)  # drops wet-market-only rows
    .withColumn("lon", F.get_json_object("geometry", "$.coordinates[0]").cast("double"))
    .withColumn("lat", F.get_json_object("geometry", "$.coordinates[1]").cast("double"))
    .withColumn("opening_year", F.regexp_extract("est_original_completion_date", r"^(\d{4})", 1))
    .withColumn("opening_year", F.when(F.col("opening_year") == "", None).otherwise(F.col("opening_year").cast("int")))
    .withColumn("is_pre_war", F.coalesce(F.lower("est_original_completion_date").contains("pre-war"), F.lit(False)))
    .withColumn("is_interim", F.col("status") == "Interim Centre")
    .withColumn("centre_id", F.concat(F.lit("HC_"), F.col("objectid")))
    .select("centre_id", "name", "status", "is_interim", "cooked_food_stalls", "lon", "lat",
            "opening_year", "is_pre_war", "description", "addressblockhousenumber",
            "addressstreetname", "addresspostalcode", "info_on_co_locators")
)

centres.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.centres")
print("centres:", centres.count())

In [0]:
c= spark.table(f"{CATALOG}.silver.centres")
print("duplicate ids:", c.count() - c.select("centre_id").distinct().count())
print("missing coords:", c.filter(F.col("lon").isNull() | F.col("lat").isNull()).count())
print("outside Singapore:", c.filter(~F.col("lon").between(103.6, 104.1) | ~F.col("lat").between(1.2, 1.48)).count())
display(c.filter(F.col("opening_year").isNull()).select("name", "is_pre_war"))

In [0]:
import json
from collections import Counter, defaultdict
import pandas as pd
from shapely.geometry import shape

MIN_JACCARD= 0.8

bld= spark.table(f"{CATALOG}.bronze.raw_hdb_buildings").select("blk_no", "st_cod", "postal_cod", "geometry").toPandas()
prop= spark.table(f"{CATALOG}.bronze.raw_hdb_property").drop("source_url", "downloaded_at", "ingested_at").toPandas()

bld["blk_no"]= bld["blk_no"].str.strip().str.upper()
prop["blk_no"]= prop["blk_no"].str.strip().str.upper()
prop["street"]= prop["street"].str.strip().str.upper()

# one point per building, guaranteed inside the footprint (a centroid can fall outside an L-shaped block)
pts= bld["geometry"].map(lambda g: shape(json.loads(g)).representative_point())
bld["lon"]= pts.map(lambda p: p.x)
bld["lat"]= pts.map(lambda p: p.y)

# score every candidate street per code by Jaccard overlap of block numbers; keep all tied for best
code_blks= bld.groupby("st_cod")["blk_no"].apply(set)
street_blks= prop.groupby("street")["blk_no"].apply(set)
blk_to_streets= defaultdict(set)
for street, blks in street_blks.items():
    for b in blks:
        blk_to_streets[b].add(street)

cands= []
for code, blks in code_blks.items():
    overlap= Counter(s for b in blks for s in blk_to_streets.get(b, ()))
    if not overlap:
        continue
    scores= {s: inter / (len(blks) + len(street_blks[s]) - inter) for s, inter in overlap.items()}
    best_j= max(scores.values())
    cands.append((code, [s for s, j in scores.items() if j == best_j], best_j))

# break ties by location: pick the street whose town is nearest to the code's buildings
street_town= prop.groupby("street")["bldg_contract_town"].agg(lambda s: s.mode()[0])
code_xy= bld.groupby("st_cod")[["lon", "lat"]].mean()
clear= {code: tied[0] for code, tied, j in cands if len(tied) == 1 and j >= MIN_JACCARD}
town_xy= (code_xy.loc[list(clear)]
          .assign(town=[street_town[clear[c]] for c in clear])
          .groupby("town")[["lon", "lat"]].mean())

def nearest_street(code, tied):
    x, y= code_xy.loc[code, "lon"], code_xy.loc[code, "lat"]
    def dist(s):
        t= street_town[s]
        if t not in town_xy.index:
            return float("inf")
        return (town_xy.loc[t, "lon"] - x)**2 + (town_xy.loc[t, "lat"] - y)**2
    return min(tied, key=dist)

rows= []
for code, tied, j in cands:
    street= tied[0] if len(tied) == 1 else nearest_street(code, tied)
    rows.append((code, street, j, len(tied)))
code_map= pd.DataFrame(rows, columns=["st_cod", "street", "jaccard", "n_tied"])

good= code_map[code_map["jaccard"] >= MIN_JACCARD]
blocks= bld.merge(good[["st_cod", "street"]], on="st_cod").merge(prop, on=["blk_no", "street"])
res= prop[prop["residential"] == "Y"]
blocks_res= blocks[blocks["residential"] == "Y"]

print("street codes:", len(code_map), "| matched at >=", MIN_JACCARD, ":", len(good), "| ties resolved by location:", (good["n_tied"] > 1).sum())
print("residential blocks:", len(res), "| matched to a building:", len(blocks_res))
print("duplicate blk+street:", blocks_res.duplicated(["blk_no", "street"]).sum())
display(good[good["n_tied"] > 1].merge(bld.groupby("st_cod")["postal_cod"].first().reset_index(), on="st_cod"))

In [0]:
# a property row matched by two buildings: keep the one nearest its street's town, drop the other
blocks_res= blocks_res.copy()
blocks_res["town_lon"]= blocks_res["street"].map(street_town).map(town_xy["lon"])
blocks_res["town_lat"]= blocks_res["street"].map(street_town).map(town_xy["lat"])
blocks_res["d_town"]= (blocks_res["lon"] - blocks_res["town_lon"])**2 + (blocks_res["lat"] - blocks_res["town_lat"])**2
dropped= blocks_res[blocks_res.duplicated(["blk_no", "street"], keep=False)].sort_values("d_town").iloc[1:]
blocks_res= (blocks_res.sort_values("d_town")
             .drop_duplicates(["blk_no", "street"])
             .drop(columns=["town_lon", "town_lat", "d_town"]))
if len(dropped):
    display(dropped[["blk_no", "street", "st_cod", "postal_cod", "lon", "lat"]])
print("duplicate blk+street:", blocks_res.duplicated(["blk_no", "street"]).sum(), "| blocks:", len(blocks_res))

In [0]:
NUM_COLS= ["max_floor_lvl", "year_completed", "total_dwelling_units",
           "1room_sold", "2room_sold", "3room_sold", "4room_sold", "5room_sold", "exec_sold",
           "multigen_sold", "studio_apartment_sold",
           "1room_rental", "2room_rental", "3room_rental", "other_room_rental"]

out= blocks_res.copy()
for c in NUM_COLS:
    out[c]= out[c].astype(int)
out["block_id"]= "B_" + out["postal_cod"].str.strip()
out= out[["block_id", "blk_no", "street", "st_cod", "postal_cod", "bldg_contract_town", "lon", "lat"]
         + NUM_COLS + ["commercial", "market_hawker", "multistorey_carpark"]]

spark.createDataFrame(out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.blocks")
print("blocks written:", len(out))

In [0]:
b= spark.table(f"{CATALOG}.silver.blocks")
print("rows:", b.count(), "| distinct block_id:", b.select("block_id").distinct().count())
print("outside Singapore:", b.filter(~F.col("lon").between(103.6, 104.1) | ~F.col("lat").between(1.2, 1.48)).count())
print("total dwelling units:", b.agg(F.sum("total_dwelling_units")).first()[0])

In [0]:
import json
import shapely
from shapely.geometry import shape, mapping

# 1. load subzones, repair invalid polygons
sz= spark.table(f"{CATALOG}.bronze.raw_subzones").select("subzone_n", "subzone_c", "pln_area_n", "pln_area_c", "region_n", "geometry").toPandas()
geoms= sz["geometry"].map(lambda g: shape(json.loads(g)))
print("invalid polygons repaired:", (~geoms.map(lambda g: g.is_valid)).sum())
sz["geom"]= geoms.map(shapely.make_valid)

sz_out= sz[["subzone_n", "subzone_c", "pln_area_n", "pln_area_c", "region_n"]].copy()
sz_out["geometry"]= sz["geom"].map(lambda g: json.dumps(mapping(g)))
spark.createDataFrame(sz_out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.subzones")

# 2. point-in-polygon: which subzone each point falls in
tree= shapely.STRtree(sz["geom"].tolist())

def tag(pdf):
    pts= shapely.points(pdf["lon"].to_numpy(), pdf["lat"].to_numpy())
    pt_idx, sz_idx= tree.query(pts, predicate="intersects")
    hit= pd.DataFrame({"i": pt_idx, "j": sz_idx}).drop_duplicates("i")  # point on a boundary: take first
    pdf= pdf.reset_index(drop=True)
    for col in ["subzone_n", "pln_area_n", "region_n"]:
        pdf[col]= None
        pdf.loc[hit["i"].to_numpy(), col]= sz[col].to_numpy()[hit["j"].to_numpy()]
    return pdf

# 3. add subzone columns to centres and blocks
for t, key in [("centres", "centre_id"), ("blocks", "block_id")]:
    sdf= spark.table(f"{CATALOG}.silver.{t}").drop("subzone_n", "pln_area_n", "region_n")
    tags= tag(sdf.select(key, "lon", "lat").toPandas())[[key, "subzone_n", "pln_area_n", "region_n"]]
    tags_sdf= spark.createDataFrame(tags, schema=f"{key} string, subzone_n string, pln_area_n string, region_n string")
    tagged= sdf.join(tags_sdf, key, "left")
    tagged.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.{t}")
    print(f"{t}: rows {tagged.count()} | no subzone {tagged.filter(F.col('subzone_n').isNull()).count()} | planning areas {tagged.select('pln_area_n').distinct().count()}")

In [0]:
import math
import numpy as np
from functools import reduce

raw_cl= spark.table(f"{CATALOG}.bronze.raw_closures")

def parse(c):
    # "nil" / "NA" / blank -> null; d/m/yyyy text -> date; anything unparseable -> null instead of crashing
    return F.expr(f"try_to_date(nullif(nullif(lower(trim({c})), 'nil'), 'na'), 'd/M/yyyy')")

base= ["serial_no", "name", "latitude_hc", "longitude_hc", "no_of_food_stalls"]
parts= [raw_cl.select(*base, F.lit("cleaning").alias("type"),
                      parse(f"q{q}_cleaningstartdate").alias("start_date"),
                      parse(f"q{q}_cleaningenddate").alias("end_date"),
                      F.col(f"remarks_q{q}").alias("remarks"))
        for q in [1, 2, 3, 4]]
parts.append(raw_cl.select(*base, F.lit("other_works").alias("type"),
                           parse("other_works_startdate").alias("start_date"),
                           parse("other_works_enddate").alias("end_date"),
                           F.col("remarks_other_works").alias("remarks")))

r= F.lower(F.coalesce(F.col("remarks"), F.lit("")))
events= (reduce(lambda a, b: a.unionByName(b), parts)
    .filter(F.col("start_date").isNotNull())
    .withColumn("type", F.when(F.col("type") == "cleaning", "cleaning")
                         .when(r.contains("commence"), "not_closure")
                         .when(r.contains("redevelop"), "redevelopment")
                         .when(r.contains("gas"), "partial")
                         .otherwise("renovation"))
    .filter(F.col("type") != "not_closure"))

# match each closures-file centre to the nearest silver centre (haversine, metres)
ev= events.toPandas()
cen= spark.table(f"{CATALOG}.silver.centres").select("centre_id", "name", "lon", "lat").toPandas()
clat, clon= np.radians(cen["lat"].to_numpy()), np.radians(cen["lon"].to_numpy())

def nearest(lat, lon):
    lat, lon= math.radians(lat), math.radians(lon)
    a= np.sin((clat - lat) / 2)**2 + np.cos(lat) * np.cos(clat) * np.sin((clon - lon) / 2)**2
    d= 2 * 6371000 * np.arcsin(np.sqrt(a))
    i= d.argmin()
    return cen["centre_id"].iat[i], cen["name"].iat[i], d[i]

loc= ev.drop_duplicates("serial_no")[base].copy()
loc[["centre_id", "centre_name", "dist_m"]]= loc.apply(
    lambda row: nearest(float(row["latitude_hc"]), float(row["longitude_hc"])), axis=1, result_type="expand")
loc["food_stalls"]= pd.to_numeric(loc["no_of_food_stalls"], errors="coerce")
same_name= loc["name"].str.upper() == loc["centre_name"].str.upper()
ok= (same_name & (loc["dist_m"] <= 200)) | (~same_name & (loc["dist_m"] <= 100) & (loc["food_stalls"] > 0))
loc.loc[~ok, "centre_id"]= None  # drops wet markets next door + Bukit Timah Market
print("events:", len(ev))
print(ev["type"].value_counts().to_string())
print("closure-file centres:", len(loc), "| matched within 200 m:", loc["centre_id"].notna().sum())
display(loc[loc["centre_id"].isna() | (loc["name"].str.upper() != loc["centre_name"].str.upper())]
        [["name", "centre_name","centre_id", "dist_m"]].sort_values("dist_m"))

In [0]:
map_sdf= spark.createDataFrame(loc[["serial_no", "centre_id"]].dropna(), schema="serial_no string, centre_id string")
closures= (events.join(map_sdf, "serial_no")
    .withColumn("duration_days", F.datediff("end_date", "start_date") + 1)
    .withColumn("closure_id", F.sha2(F.concat_ws("|", "centre_id", "type", "start_date"), 256).substr(1, 12))
    .select("closure_id", "centre_id", "type", "start_date", "end_date", "duration_days", "remarks"))
closures.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.closures")

c= spark.table(f"{CATALOG}.silver.closures")
print("rows:", c.count(), "| duplicate ids:", c.count() - c.select("closure_id").distinct().count(),
      "| end before start:", c.filter(F.col("end_date") < F.col("start_date")).count())
display(c.filter(F.col("type") != "cleaning")
        .join(spark.table(f"{CATALOG}.silver.centres").select("centre_id", "name"), "centre_id")
        .orderBy("start_date"))

In [0]:
# --- seniors + residents per block, from June 2026 subzone x flat type counts ---
HDB_TYPES= {"1- and 2-Room Flats*": "r12", "3-Room Flats": "r3", "4-Room Flats": "r4", "5-Room and Executive Flats": "r5"}
SENIOR_AGES= ["65 - 69", "70 - 74", "75 - 79", "80 - 84", "85 - 89", "90 & Over"]

pop= spark.table(f"{CATALOG}.bronze.raw_population_2026").filter(F.col("sheet") == "2026(Total)").toPandas()
pop= pop[(pop["subzone"] != "") & (pop["subzone"] != "Total") & (pop["age_group"] != "Total")]  # drops footnotes + roll-ups
pop= pop[pop["type_of_dwelling"].isin(HDB_TYPES)]
pop["residents"]= pd.to_numeric(pop["2026"].replace("-", "0")).astype(int)  # "-" = nil or negligible
pop["flat_type"]= pop["type_of_dwelling"].map(HDB_TYPES)
pop["subzone_n"]= pop["subzone"].str.upper().str.strip()
pop["seniors"]= pop["residents"].where(pop["age_group"].isin(SENIOR_AGES), 0)
sz_pop= pop.groupby(["subzone_n", "flat_type"], as_index=False)[["residents", "seniors"]].sum()
print("file totals | HDB residents:", sz_pop["residents"].sum(), "| HDB 65+:", sz_pop["seniors"].sum())

blk= spark.table(f"{CATALOG}.silver.blocks").toPandas()
blk["r12"]= blk[["1room_sold", "2room_sold", "studio_apartment_sold", "1room_rental", "2room_rental"]].sum(axis=1)
blk["r3"]= blk[["3room_sold", "3room_rental"]].sum(axis=1)
blk["r4"]= blk[["4room_sold", "other_room_rental"]].sum(axis=1)  # assumption: other rental counted as 4-room
blk["r5"]= blk[["5room_sold", "exec_sold", "multigen_sold"]].sum(axis=1)
blk["rental_units"]= blk[["1room_rental", "2room_rental", "3room_rental", "other_room_rental"]].sum(axis=1)

units= blk.melt(id_vars=["block_id", "subzone_n"], value_vars=["r12", "r3", "r4", "r5"], var_name="flat_type", value_name="units")
sz_units= units.groupby(["subzone_n", "flat_type"], as_index=False)["units"].sum()
rates= sz_pop.merge(sz_units, on=["subzone_n", "flat_type"], how="outer")
rates[["residents", "seniors", "units"]]= rates[["residents", "seniors", "units"]].fillna(0)
rates["seniors_per_unit"]= np.where(rates["units"] > 0, rates["seniors"] / rates["units"], 0)
rates["residents_per_unit"]= np.where(rates["units"] > 0, rates["residents"] / rates["units"], 0)

lost= rates[(rates["units"] == 0) & (rates["seniors"] > 0)]
print("seniors with no matching flats in our blocks:", int(lost["seniors"].sum()), "across", lost["subzone_n"].nunique(), "subzones")

u= units.merge(rates[["subzone_n", "flat_type", "seniors_per_unit", "residents_per_unit"]], on=["subzone_n", "flat_type"], how="left").fillna(0)
u["est_seniors"]= u["units"] * u["seniors_per_unit"]
u["est_residents"]= u["units"] * u["residents_per_unit"]
blk_pop= u.groupby("block_id", as_index=False)[["est_seniors", "est_residents"]].sum().merge(blk[["block_id", "rental_units"]], on="block_id")

pop_sdf= spark.createDataFrame(blk_pop, schema="block_id string, est_seniors double, est_residents double, rental_units long")
blocks_sdf= spark.table(f"{CATALOG}.silver.blocks").drop("est_seniors", "est_residents", "rental_units")
blocks_sdf.join(pop_sdf, "block_id", "left").write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.blocks")

b= spark.table(f"{CATALOG}.silver.blocks")
print("blocks:", b.count(), "| est seniors:", round(b.agg(F.sum("est_seniors")).first()[0]), "| est residents:", round(b.agg(F.sum("est_residents")).first()[0]))
display(b.groupBy("pln_area_n").agg(F.round(F.sum("est_seniors")).alias("seniors"), F.round(F.sum("est_residents")).alias("residents"))
        .orderBy(F.desc("seniors")).limit(10))